# Titanic Dataset ML Project
Dataset from Kaggle [Titanic Challenge](https://www.kaggle.com/competitions/titanic/data)

## Setup

### Import Packages

In [1]:
# Include tqdm for monitoring progress as things run
import matplotlib.pyplot as plt
import sklearn as sk
import pandas as pd
import numpy as np
import xgboost
import tqdm
import re

## Dataset

### Import Data

In [2]:
# use pandas
DATA = pd.read_csv('data/train.csv')

In [3]:
#print(DATA.head())
y_train = DATA['Survived']
#y_train

In [4]:
x_train = DATA.drop(['Survived','PassengerId','Ticket',],axis=1)
#x_train.head()

In [5]:
x_train.dtypes

Pclass        int64
Name            str
Sex             str
Age         float64
SibSp         int64
Parch         int64
Fare        float64
Cabin           str
Embarked        str
dtype: object

### Clean Dataset Check

In [6]:
check_empties = []
for column in x_train.columns:
    tmp_list = [column, int(np.sum(x_train[column].isna()))]
    #print(column)
    #print(np.sum(x_train[column].isna()))
    check_empties.append(tmp_list)
print(check_empties)

[['Pclass', 0], ['Name', 0], ['Sex', 0], ['Age', 177], ['SibSp', 0], ['Parch', 0], ['Fare', 0], ['Cabin', 687], ['Embarked', 2]]


In [7]:
category = 'Name'
for value in zip(x_train[category],x_train[category].isna()):
    print(value)
#print(enumerate(x_train['Cabin'].isna())

('Braund, Mr. Owen Harris', False)
('Cumings, Mrs. John Bradley (Florence Briggs Thayer)', False)
('Heikkinen, Miss. Laina', False)
('Futrelle, Mrs. Jacques Heath (Lily May Peel)', False)
('Allen, Mr. William Henry', False)
('Moran, Mr. James', False)
('McCarthy, Mr. Timothy J', False)
('Palsson, Master. Gosta Leonard', False)
('Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)', False)
('Nasser, Mrs. Nicholas (Adele Achem)', False)
('Sandstrom, Miss. Marguerite Rut', False)
('Bonnell, Miss. Elizabeth', False)
('Saundercock, Mr. William Henry', False)
('Andersson, Mr. Anders Johan', False)
('Vestrom, Miss. Hulda Amanda Adolfina', False)
('Hewlett, Mrs. (Mary D Kingcome) ', False)
('Rice, Master. Eugene', False)
('Williams, Mr. Charles Eugene', False)
('Vander Planke, Mrs. Julius (Emelia Maria Vandemoortele)', False)
('Masselmani, Mrs. Fatima', False)
('Fynney, Mr. Joseph J', False)
('Beesley, Mr. Lawrence', False)
('McGowan, Miss. Anna "Annie"', False)
('Sloper, Mr. William Thompson', 

### Imputing Values Required?

#### Imputing Age

In [8]:
# Impute missing age values as median of Name title (Mr. Mrs. Ms. ...), Pclass (1, 2, 3), and Sex (Male, Female)

In [9]:
# Find all titles: Assume title starts after ", " and ends after ".", no missing names in dataset

# Strip all names down to title/honorific using regex
Titles = []
for name in x_train['Name']: # For all names in the dataset
    match = re.search(", .*?\\. ",name).group() # The part of the string that matches our conditions to start at ', ' and end at '. '
    Titles.append(match.lstrip(', ').rstrip(' ')) # Append the honorific stripped of prefix/suffix

UTitles = set(Titles) # All unique titles
print(len(UTitles)," unique titles / honorifics")
print(UTitles)

17  unique titles / honorifics
{'Rev.', 'Sir.', 'Master.', 'Dr.', 'Mr.', 'Ms.', 'the Countess.', 'Capt.', 'Don.', 'Miss.', 'Major.', 'Lady.', 'Jonkheer.', 'Mme.', 'Mlle.', 'Col.', 'Mrs.'}


In [10]:
# Imputes age values by sex, social status, then by title/honorific
unique_titles = x_train['Name'].str.extract(f"({'|'.join(UTitles)})", expand=False) # Searching for any titles in the unique list

x_train['Age'] = x_train['Age'].fillna(x_train.groupby(['Sex', 'Pclass', unique_titles])['Age'].transform('median'))

In [11]:
# If no other examples within sex/social status/title exist, impute with global median
global_median = x_train['Age'].median()
x_train['Age'] = x_train['Age'].fillna(global_median)

#### Imputing Cabin (removing numeric vals, keeping alpha designation)

In [12]:
# Not implemented yet

### Encoding Required?

In [13]:
x_train = x_train.drop(columns=['Name','Cabin'])

In [14]:
x_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


In [24]:
# Encode sex with pd dummies

# Imputing embarked categorical with most often occuring value, no missing values for sex (excluded)
x_train['Embarked'] = x_train['Embarked'].fillna(x_train['Embarked'].mode()[0]) 

# Convert cleans categoricals to OHE cols
x_train_encoded = pd.get_dummies(x_train, columns=['Embarked','Sex'], drop_first=True)

# Convert T/F to 1/0
x_train_encoded = x_train_encoded.astype(float)


In [25]:
# Depends on the model used, not for RF/XGB
# Converting string columns to categories
# Use OHE

#x_train_encoded = pd.get_dummies(x_train,drop_first=True)

#for col in x_train.columns:
#    if (x_train[col].dtype == str) or (x_train[col].dtype == object):
#        print(col)
#        x_train[col] = x_train[col].astype('category')


### Train Test Split

In [26]:
# Not necessary, separate file given 

### Shuffling Training Examples

In [27]:
x_train_encoded, y_train = sk.utils.shuffle(x_train_encoded,y_train,random_state=12)

In [28]:
x_train_encoded.head()

,Pclass,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S,Sex_male
456,1.0,65.0,0.0,0.0,26.5500,0.0,1.0,1.0
351,1.0,39.5,0.0,0.0,35.0000,0.0,1.0,1.0
173,3.0,21.0,0.0,0.0,7.9250,0.0,1.0,1.0
671,1.0,31.0,1.0,0.0,52.0000,0.0,1.0,1.0
836,3.0,21.0,0.0,0.0,8.6625,0.0,1.0,1.0


In [29]:
y_train.head() # checking if examples still ordered wrt x_train_shuff; verified

886    0
13     0
97     1
410    0
577    1
Name: Survived, dtype: int64

### Imputing Values if Required

## Model

### Model Setup

In [30]:
# Models used (Estimators dict / list)
# Use: Random Forest, XGBoost, LightGBM (Not the last one)
estimators = {
    #[name, model]
    'random_forest':   sk.ensemble.RandomForestClassifier(),
    'xgboost':         xgboost.XGBClassifier() 
    }

In [31]:
estimators.items()

dict_items([('random_forest', RandomForestClassifier()), ('xgboost', XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...))])

### Training

In [33]:
# model.fit(x_train,y_train)
models = {}

for model_name, model in estimators.items():
    #print(model_name)
    trained = model.fit(x_train_encoded,y_train)
    models[model_name] = trained
    trained = None
    #print(model)

### Initial Results

In [50]:
# Visualized; accuracy, PR Curves, AUC table summary
# model.predict(x_test)
# sklearn.metrics.score(y_train,y_test)

print("Where 0 represents an individual passenger's survival "
        "and 1 represents a single passenger not surviving:\n")

for model_name in models:
    y_pred = models[model_name].predict(x_train_encoded)
    class_rep = sk.metrics.classification_report(y_train,y_pred)
    print(f"Classification Report for {model_name.replace("_"," ").capitalize()} model:\n{class_rep}")
    accuracy = sk.metrics.accuracy_score(y_train, y_pred)
    #AUC = sk.metrics.auc(y_train,y_pred)

    #
    # print("Accuracy of",model_name,"model: ",accuracy)
    #print("AUC of",model_name,"model: ",AUC)

Where 0 represents an individual passenger's survival and 1 represents a single passenger not surviving:

Classification Report for Random forest model:
              precision    recall  f1-score   support

           0       0.94      0.95      0.95       549
           1       0.93      0.90      0.91       342

    accuracy                           0.93       891
   macro avg       0.93      0.93      0.93       891
weighted avg       0.93      0.93      0.93       891

Classification Report for Xgboost model:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92       549
           1       0.89      0.83      0.86       342

    accuracy                           0.90       891
   macro avg       0.90      0.88      0.89       891
weighted avg       0.90      0.90      0.90       891



## Optimization

### Optuna Objective

### Train & Validation Sets

### Optimize Against Validation Set

### Perfomance Versus Unoptimized

### Results